![UTN Facultad Regional Mendoza](https://raw.githubusercontent.com/javovelez/Modelos-de-Lenguaje/main/img/logo_utn_frm.png)

# Laboratorio n° 3. Parte A. Reconocimiento de entidades con BERT

**Asignatura:** Modelos de Lenguaje
**Bloque:** 3. Atención y Transformers

---

## Introducción

El clasificador de reseñas de la Clase 2 de esta unidad, `ClasificadorBERT`, toma de la salida de BETO un solo vector, el de la posición `[CLS]`, y le aplica una capa lineal que produce cinco puntajes, uno por cantidad de estrellas. Los vectores de las demás posiciones no intervienen en la predicción. La Figura 1 del artículo de BERT, que la Clase 2 reproduce en la sección 1.3, muestra otras cabezas sobre el mismo encoder. Una de ellas aplica la capa lineal al vector de cada posición y produce una etiqueta por pieza (subpalabra). Este laboratorio construye esa cabeza y la afina.

La tarea es el **reconocimiento de entidades nombradas**, en inglés *named entity recognition* o NER. Una entidad nombrada es una expresión que designa a una persona, una organización o un lugar, como «José Palomares», «Real Madrid» o «Caracas». Un etiquetador de entidades asigna a cada palabra de una oración una etiqueta que indica si la palabra forma parte de una entidad y de qué tipo. El corpus es la parte en español de CoNLL-2002, un conjunto de oraciones de una agencia de noticias española, etiquetadas a mano.

Respecto de la Clase 2, el laboratorio cambia la unidad de la etiqueta, la métrica y las comparaciones.

- **La unidad de la etiqueta.** CoNLL-2002 trae una etiqueta por palabra, y BETO procesa piezas, como mostró la sección 1.4 de la Clase 2. Antes de afinar hay que trasladar cada etiqueta de la palabra a sus piezas, y después de predecir hay que hacer el camino inverso.
- **La métrica.** El laboratorio mide con el **F1 por entidad**, que cuenta entidades completas en lugar de palabras. El Ejercicio 1 compara esa métrica con la accuracy por palabra.
- **Las comparaciones.** El etiquetador se compara con dos líneas de base que no aprenden, con la clase equivalente de Hugging Face y con el mismo etiquetador construido sobre el BERT multilingüe. El último ejercicio verifica si las particiones de evaluación comparten oraciones con la de entrenamiento.

Al completar este laboratorio vas a poder:

- Explicar el esquema de etiquetas BIO y medir un etiquetador con el F1 por entidad.
- Trasladar las etiquetas de las palabras a las piezas de un tokenizador, y las predicciones de las piezas a las palabras.
- Escribir una cabeza de clasificación por posición sobre el encoder de BETO.
- Explicar por qué una representación contextual distingue dos usos de una misma palabra que una línea de base por memorización confunde.
- Elegir entre dos modelos preentrenados afinados para la misma tarea, con la variación entre semillas como escala para juzgar la diferencia entre los dos.
- Verificar si las particiones de evaluación de un corpus comparten ejemplos con la de entrenamiento, y medir cuánto cambia la cifra de evaluación sin esos ejemplos.

---

## Instrucciones generales

- Completá el código en las celdas marcadas con `# Tu código aquí`.
- Respondé las preguntas de análisis en las celdas de texto (tipo Markdown).
- **El laboratorio necesita GPU.** En Colab, *Entorno de ejecución > Cambiar tipo de entorno > T4 GPU*. El laboratorio afina cinco modelos de entre 110 y 180 millones de parámetros, tres épocas cada uno. Con una T4, el total se estima del orden de diez minutos. Antes de empezar, verificá que la primera celda de preparación imprima `cuda` como dispositivo.
- **La primera celda de preparación pide acceso a Google Drive.** La celda crea en Drive la carpeta `modelos-lenguaje-u3`, donde se guarda el resultado de cada afinado. Cada resultado incluye la configuración, la curva de la pérdida y las predicciones, y no incluye los pesos del modelo, que ocupan unos 440 MB. Si la sesión se interrumpe, al volver a ejecutar las celdas los afinados terminados se cargan de esa carpeta y no se repiten. Si no autorizás el acceso, la carpeta se crea en el disco de la sesión y se pierde al cerrarla.
- Este laboratorio va después de la Clase 2 de la Unidad 3. Da por vistos Hugging Face, el tokenizador de BETO y sus piezas (sección 1.4), `ClasificadorBERT` y la receta de afinado (sección 3.1). El reconocimiento de entidades, sus etiquetas y su métrica no están en la clase, y el laboratorio los explica antes de pedirlos.
- Usá las semillas que pide cada enunciado.
- Las cifras de un afinado varían entre máquinas y entre corridas, por los motivos que explica la sección «Sobre los números» de la Clase 2. Lo que se corrige es la dirección de cada resultado y el argumento con el que lo explicás.

## IMPORTANTE. Qué celdas podés modificar

Este laboratorio es un **entregable**. Solo debés completar las celdas de actividad, que son las que aparecen con el comentario `# Tu código aquí` o el texto `*(Escribí tu respuesta acá)*`. Las demás celdas, es decir, los enunciados, las explicaciones, los ejemplos provistos y el encabezado, **no se tocan**.

La corrección se hace celda por celda, y cada respuesta se busca en la celda donde el enunciado la pide. Si escribís en otro lado, o si movés, renombrás o borrás celdas del enunciado, esa parte de tu entrega queda sin poder corregirse.

Si querés probar algo por separado, hacelo en la misma celda de actividad o en una celda nueva que agregues, y borrala antes de entregar.

---
## Preparación

Las tres celdas que siguen vienen resueltas. No hay nada que completar en ellas, pero hay que ejecutarlas en orden antes de empezar. Cada una lleva antes una presentación con los nombres que deja definidos, y los ejercicios usan esos nombres.

La primera celda instala las bibliotecas que falten, las importa y prepara la carpeta de resultados. La segunda descarga el corpus CoNLL-2002. La tercera define la función que mide un etiquetador.

El laboratorio tiene cuatro celdas de preparación más, ubicadas junto a los ejercicios que las necesitan. La primera, antes del Ejercicio 2, aplica el tokenizador de BETO a una oración ya dividida en palabras. La segunda, antes del Ejercicio 3, define el bucle de afinado. La tercera, dentro del Ejercicio 3, afina los cuatro modelos sobre BETO, porque necesita la clase que escribís en ese ejercicio. La cuarta, también dentro del Ejercicio 3, cuenta los aciertos sobre la palabra «Madrid», porque necesita esos modelos afinados.

| Nombre | Qué es |
|---|---|
| `DISPOSITIVO` | `"cuda"` si hay GPU; en otro caso, `"mps"` o `"cpu"` |
| `EN_GPU` | `True` si el dispositivo es CUDA. Activa la precisión mixta |
| `BETO`, `MULTI` | los nombres en el Hub de BETO y del BERT multilingüe, los mismos de la Clase 2 |
| `CARPETA` | la carpeta donde se guardan los resultados de los afinados |
| `descargar` | descarga un archivo del laboratorio, salvo que ya exista en el disco de la sesión |

In [ ]:
# ─── Preparación. Bibliotecas, módulo de la materia y carpeta de resultados ─
import os, sys, math, time, json, hashlib, importlib, subprocess, urllib.request, warnings
from collections import Counter
from importlib.metadata import version, PackageNotFoundError


def instalar(*argumentos):
    """Instala con el pip del intérprete del notebook. Si pip falla, corta y muestra su error."""
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *argumentos],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"pip install {' '.join(argumentos)} falló:\n{r.stderr[-3000:]}")
    importlib.invalidate_caches()                     # para que el import vea el paquete recién instalado


try:                                                  # transformers 5 o posterior, como la Clase 2
    mayor = int(version("transformers").split(".")[0])
except PackageNotFoundError:
    mayor = 0
if mayor < 5:
    instalar("-U", "transformers>=5")
try:
    import seqeval
except ImportError:
    instalar("seqeval")

os.environ.setdefault("HF_HUB_VERBOSITY", "error")
warnings.filterwarnings("ignore", message="Detected call of `lr_scheduler.step")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
from transformers import AutoTokenizer, BertModel, BertForTokenClassification
from seqeval.metrics import classification_report
from seqeval.metrics.sequence_labeling import get_entities

transformers.logging.set_verbosity_error()
transformers.utils.logging.disable_progress_bar()
pd.set_option("display.max_columns", 40)            # las tablas de una oración, enteras
pd.set_option("display.width", 250)

# El repositorio de la materia da una URL estable. El Drive de la cátedra queda de
# respaldo para el módulo, porque Drive corta las descargas públicas por cuota.
RELEASES = "https://github.com/javovelez/Modelos-de-Lenguaje/releases/download"
CONLL = ("https://huggingface.co/datasets/eriktks/conll2002/resolve/"
         "refs%2Fconvert%2Fparquet/es/{}/0000.parquet")
FUENTES = {                                     # archivo: (URL, ID de Drive)
    "cache_modelos.py":         (f"{RELEASES}/utils-v1/cache_modelos.py",
                                 "1k5wxbSMS-_3dC8FN_F-2VN6dp8tGFCpz"),
    "conll_train.parquet":      (CONLL.format("train"), None),
    "conll_validation.parquet": (CONLL.format("validation"), None),
    "conll_test.parquet":       (CONLL.format("test"), None),
}


def descargar(nombre):
    """Descarga un archivo de FUENTES, salvo que ya exista. Si la URL no responde, usa Drive."""
    if os.path.exists(nombre):
        return
    url, id_drive = FUENTES[nombre]
    try:
        urllib.request.urlretrieve(url, nombre)
    except Exception as falla:
        if id_drive is None:
            raise
        print(f"el repositorio de la materia no respondió ({falla}); se usa el Drive.")
        try:
            import gdown
        except ImportError:
            instalar("gdown")
            import gdown
        gdown.download(id=id_drive, output=nombre, quiet=False)


descargar("cache_modelos.py")
from cache_modelos import preparar_carpeta

if torch.cuda.is_available():
    DISPOSITIVO = "cuda"
elif torch.backends.mps.is_available():
    DISPOSITIVO = "mps"
else:
    DISPOSITIVO = "cpu"
EN_GPU = DISPOSITIVO == "cuda"                      # la precisión mixta se usa solo en CUDA

BETO = "dccuchile/bert-base-spanish-wwm-cased"      # el BERT en español
MULTI = "bert-base-multilingual-cased"              # el BERT original, en 104 idiomas
CARPETA = preparar_carpeta("modelos-lenguaje-u3")
torch.manual_seed(0)
if DISPOSITIVO == "cpu":
    print("Sin GPU. Cambiá el tipo de entorno a T4 antes de seguir.")

pd.Series({"transformers": transformers.__version__, "torch": torch.__version__,
           "dispositivo": DISPOSITIVO}).to_frame("versión")

### El corpus CoNLL-2002

CoNLL-2002 es un corpus de reconocimiento de entidades publicado en 2002. La parte en español reúne oraciones de una agencia de noticias española. Cada oración viene dividida en palabras, con los signos de puntuación como palabras separadas, y cada palabra tiene una etiqueta. Las particiones de entrenamiento, validación y prueba vienen fijadas por el corpus. La celda las descarga del Hub de Hugging Face en formato *parquet*, y las tres ocupan 1,7 MB.

Cada etiqueta indica si la palabra forma parte de una entidad y, en ese caso, de qué tipo es la entidad. CoNLL-2002 distingue cuatro tipos, `PER` para personas, como «José María Aznar», `ORG` para organizaciones, como «Real Madrid», `LOC` para lugares, como «Santiago de Compostela», y `MISC`, de misceláneo, para las entidades que no pertenecen a los otros tres, como eventos, obras o gentilicios. «Liga de Campeones» es una entidad de tipo `MISC`. Una palabra que no forma parte de ninguna entidad tiene la etiqueta `O`, del inglés *outside*.

Una entidad puede ocupar varias palabras. Para marcar en qué palabra empieza cada entidad, las etiquetas siguen el **esquema BIO**. La etiqueta de la primera palabra de una entidad es el tipo precedido por `B-`, del inglés *begin*. La etiqueta de cada palabra siguiente es el tipo precedido por `I-`, de *inside*. Por ejemplo, «Santiago de Compostela» lleva las etiquetas `B-LOC I-LOC I-LOC`. El prefijo separa dos entidades del mismo tipo que aparecen seguidas. En el encabezado «Cantabria Santiago de Compostela», las etiquetas `B-LOC B-LOC I-LOC I-LOC` indican dos lugares y no uno. En total, el esquema tiene nueve etiquetas, que son la `O` y las ocho combinaciones de los dos prefijos con los cuatro tipos.

La celda a continuación imprime la cantidad de oraciones, palabras y entidades de cada partición, y una oración de validación con sus etiquetas. En esa oración, «José Palomares» es una entidad de tipo persona de dos palabras, así que «José» lleva `B-PER` y «Palomares» lleva `I-PER`. «EFE», que es el nombre de la agencia, y «RSN» son dos entidades de tipo organización de una palabra cada una.

| Nombre | Qué es |
|---|---|
| `ETIQUETAS` | la lista de las nueve etiquetas. La posición de cada una es su índice, y `O` es el 0 |
| `entidades(etiquetas)` | la función que extrae las entidades de una oración. El argumento `etiquetas` es la lista de índices de etiqueta de esa oración, con un índice por palabra. La función devuelve una lista con una tupla `(tipo, primera, última)` por entidad. Los valores `primera` y `última` son las posiciones de la primera y la última palabra de la entidad, contadas desde 0. Si una oración lleva las etiquetas `O B-LOC I-LOC I-LOC`, la función devuelve `[("LOC", 1, 3)]` |
| `ent`, `val`, `prueba` | las tres particiones, como listas de pares `(palabras, etiquetas)`. Las palabras son textos y las etiquetas son índices de `ETIQUETAS` |
| `particiones` | un diccionario con las tres particiones, con las claves `"ent"`, `"val"` y `"prueba"` |
| `EJEMPLO` | la posición en `val` de la oración que imprime la celda |

In [ ]:
# ─── Preparación. El corpus CoNLL-2002 en español ──────────────────────────
ETIQUETAS = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC", "B-MISC", "I-MISC"]
TIPOS = ["PER", "ORG", "LOC", "MISC"]

particiones = {}
for clave, archivo in (("ent", "train"), ("val", "validation"), ("prueba", "test")):
    descargar(f"conll_{archivo}.parquet")
    tabla = pd.read_parquet(f"conll_{archivo}.parquet")
    particiones[clave] = [(list(f.tokens), [int(e) for e in f.ner_tags]) for f in tabla.itertuples()]
ent, val, prueba = particiones["ent"], particiones["val"], particiones["prueba"]


def entidades(etiquetas):
    """Las entidades de una oración, como tuplas (tipo, posición de la primera palabra, posición de la última)."""
    return get_entities([ETIQUETAS[e] for e in etiquetas])


def contar(oraciones):
    tipos = Counter(t for _, etq in oraciones for t, _, _ in entidades(etq))
    return {"oraciones": len(oraciones), "palabras": sum(len(p) for p, _ in oraciones),
            "entidades": sum(tipos.values()), **{t: tipos[t] for t in TIPOS}}


EJEMPLO = next(i for i, (palabras, _) in enumerate(val) if "Palomares" in palabras)

display(pd.DataFrame({clave: contar(o) for clave, o in particiones.items()}))
pd.DataFrame({"palabra": val[EJEMPLO][0],
              "etiqueta": [ETIQUETAS[e] for e in val[EJEMPLO][1]]}).T

### El F1 por entidad

El desempeño del etiquetador se mide comparando las entidades predichas con las entidades de referencia. Una entidad queda definida por tres datos, que son su tipo, su primera palabra y su última palabra. Una entidad predicha es un **acierto** cuando una entidad de referencia tiene los mismos tres datos. Por eso una entidad reconocida a medias, por ejemplo con una palabra de menos, no es un acierto, aunque casi todas sus palabras tengan la etiqueta correcta. Con los aciertos se calculan tres cifras:

- la **precisión**, que es la proporción de entidades predichas que son aciertos;
- el *recall*, que es la proporción de entidades de referencia que el etiquetador encontró;
- el **F1**, que es la media armónica de las dos, $F_1 = 2PR / (P + R)$.

Una palabra que lleva la etiqueta `O` en la referencia y en la predicción no interviene en ninguna de las tres cifras, porque no forma una entidad en ninguna de las dos. Un error sobre esa palabra sí interviene. Si la referencia dice `O` y el etiquetador predice `B-PER`, la predicción forma una entidad `PER` que no coincide con ninguna entidad de referencia. Esa entidad baja la precisión.

La función `medir` extrae las entidades de las etiquetas de referencia y de las etiquetas predichas, y calcula las tres cifras con la biblioteca `seqeval`. Esa biblioteca aplica el mismo criterio que el programa de evaluación oficial de CoNLL-2002. Si una predicción tiene una etiqueta `I-` sin la `B-` correspondiente, ese criterio abre una entidad nueva en esa palabra. La celda verifica `medir` comparando las etiquetas de referencia de validación con sí mismas, lo que tiene que dar 100 en todas las cifras.

| Nombre | Qué es |
|---|---|
| `medir(oraciones, predichas)` | la función que calcula las tres cifras. El argumento `oraciones` es una lista de pares `(palabras, etiquetas)` con las etiquetas de referencia, con el mismo formato que `val`. El argumento `predichas` tiene una lista por oración, en el mismo orden que `oraciones`, y cada una de esas listas tiene un índice de etiqueta por palabra. La función devuelve una `Series` con el F1, la precisión, el *recall* y el F1 de cada tipo, en porcentaje |

In [ ]:
# ─── Preparación. El F1 por entidad ────────────────────────────────────────
def medir(oraciones, predichas):
    """F1 por entidad con el criterio de CoNLL (seqeval), en %.

    oraciones: lista de pares (palabras, etiquetas), con las etiquetas de referencia.
    predichas: una lista por oración, en el orden de oraciones, con un índice de etiqueta por palabra.
    Devuelve una Series con F1, precisión, recall y el F1 de cada tipo.
    """
    reales = [[ETIQUETAS[e] for e in etq] for _, etq in oraciones]
    pred = [[ETIQUETAS[e] for e in p] for p in predichas]
    r = classification_report(reales, pred, output_dict=True, zero_division=0)
    fila = {"F1": r["micro avg"]["f1-score"], "precisión": r["micro avg"]["precision"],
            "recall": r["micro avg"]["recall"]}
    for tipo in TIPOS:
        fila[f"F1 {tipo}"] = r[tipo]["f1-score"] if tipo in r else 0.0
    return (100 * pd.Series(fila)).round(1)


medir(val, [etq for _, etq in val]).to_frame("referencia contra sí misma")

### Ejercicio 1. La accuracy por palabra y el F1 por entidad

**Objetivo.** Medir dos predictores con la accuracy por palabra y con el F1 por entidad, para decidir cuál de las dos métricas sirve para comparar etiquetadores.

Los dos predictores son **líneas de base**. Una línea de base es un predictor construido con una regla fija, contra el que se mide un modelo nuevo. La primera línea de base asigna `O` a todas las palabras. La segunda asigna a cada palabra la etiqueta con la que más veces aparece en entrenamiento. Las dos se miden sobre la partición de validación. El F1 de la segunda es la cifra que el etiquetador de los ejercicios siguientes tiene que superar.

**Enunciado:**

1. Escribí la función `accuracy_por_palabra(oraciones, predichas)`. La función tiene que recibir los mismos dos argumentos que `medir` y devolver el porcentaje de palabras cuya etiqueta predicha coincide con la de referencia, contado sobre todas las palabras de todas las oraciones.
2. Creá una lista llamada `pred_O` que contenga, para cada oración de `val`, una lista de ceros del largo de la oración. Esa lista es la línea de base que predice `O` en todas las palabras.
3. Creá la línea de base de **memorización**, que asigna a cada palabra la etiqueta con la que más veces aparece en entrenamiento. La línea de base se construye en tres pasos.
   - Creá un diccionario vacío llamado `conteo`. Recorré cada oración de `ent`, y dentro de cada oración recorré cada palabra junto con su índice de etiqueta. Si la palabra todavía no es clave de `conteo`, agregala con un `Counter` vacío como valor. Después sumá 1 a la cuenta del índice de etiqueta en el `Counter` de esa palabra. Al terminar, `conteo["Madrid"]` es un `Counter` que guarda cuántas veces aparece «Madrid» con cada índice de etiqueta.
   - Construí a partir de `conteo` un diccionario llamado `memoria`. Las claves de `memoria` son las palabras de `conteo`. El valor de cada palabra es el índice de etiqueta con más apariciones en su `Counter`.
   - Creá una lista llamada `pred_memoria`, con el mismo formato que `pred_O`. Asigná a cada palabra de cada oración de `val` su índice de etiqueta en `memoria`. A las palabras que no son claves de `memoria`, asignales `0`, que es el índice de `O`.
4. Construí un `DataFrame` llamado `lineas_de_base`, con una fila por línea de base y los índices `"todo O"` y `"memorización"`. La primera columna es `"accuracy por palabra"`, y las demás son las de la `Series` que devuelve `medir`. Mostralo con un decimal.
5. Imprimí el conteo de la palabra `"Madrid"` en `conteo`, con el nombre de cada etiqueta en lugar de su índice. La pregunta de análisis del Ejercicio 3 usa ese conteo.

> **Pista.** El método `setdefault(clave, valor)` de un diccionario devuelve el valor de la clave, y antes agrega la clave con `valor` si no existe. En un `Counter`, sumar 1 a una clave que no existe la crea con cuenta 1. El método `most_common(1)` de un `Counter` devuelve una lista con el par `(elemento, cantidad)` más frecuente.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

Compará la accuracy por palabra y el F1 de la línea de base todo O. Explicá qué mide cada cifra en ese caso, y cuál de las dos sirve para comparar etiquetadores en este corpus. Usá la proporción de palabras de validación que llevan la etiqueta `O`.

*(Escribí tu respuesta acá)*

---
### Preparación. Las piezas de BETO para una oración dividida en palabras

En la Clase 2, el tokenizador de BETO recibía un texto entero y lo procesaba en dos pasos. Primero dividía el texto en palabras, separando por los espacios y por los signos de puntuación, y después dividía cada palabra en piezas (subpalabras). Las oraciones de CoNLL-2002 no vienen como texto entero. Cada oración viene como una lista de palabras. Cada etiqueta del corpus corresponde a una palabra de esa lista. La división en palabras del tokenizador no siempre coincide con la del corpus. El argumento `is_split_into_words=True` le indica al tokenizador que recibe una lista de palabras ya dividida. Con ese argumento, el tokenizador conserva las palabras del corpus y divide cada una en sus piezas.

El tokenizador procesa una oración o varias en la misma llamada. Si recibe una lista de oraciones, devuelve una secuencia de piezas por oración. Las secuencias no se rellenan, así que cada una conserva su propio largo. Con `is_split_into_words=True`, cada elemento de esa lista es la lista de palabras de una oración.

La salida del encoder de BETO tiene un vector por pieza. Las etiquetas, en cambio, corresponden a palabras. La secuencia de piezas de una oración es más larga que su lista de palabras, por dos motivos. Una palabra puede ocupar varias piezas. Además, el tokenizador agrega `[CLS]` al principio de la secuencia y `[SEP]` al final. Para pasar las etiquetas de las palabras a las piezas, el código necesita saber de qué palabra sale cada pieza.

Al dividir las palabras en piezas, el tokenizador registra de qué palabra sale cada pieza. El método `word_ids()` devuelve ese registro. En la celda siguiente, `tok_beto` procesa la oración de ejemplo y guarda el resultado en `codificada`. La llamada `codificada.word_ids()` devuelve una lista con un elemento por posición de la secuencia de piezas. Cada elemento es el índice de la palabra de la que sale la pieza, es decir, su posición en la lista de palabras, contada desde 0. Las posiciones de `[CLS]` y `[SEP]` no salen de ninguna palabra. En esas posiciones, el elemento es `None`. Cuando el tokenizador procesa varias oraciones en una sola llamada, `word_ids(i)` devuelve la lista de la oración `i`. Ni el argumento `is_split_into_words` ni el método `word_ids()` aparecen en la Clase 2.

La celda aplica el tokenizador a la oración de ejemplo y muestra, para cada posición, la pieza, el índice que devuelve `word_ids()` y la palabra correspondiente. La palabra «Palomares» ocupa tres posiciones, `Pal`, `##oma` y `##res`, y las tres tienen el mismo índice de palabra.

La celda fija además el largo máximo de una secuencia en 128 piezas, e imprime el percentil 99 del largo de las oraciones de entrenamiento y la cantidad de oraciones que superan el máximo. Para medir esos largos, la celda tokeniza todas las oraciones de `ent` en una sola llamada. Esas oraciones se truncan.

| Nombre | Qué es |
|---|---|
| `tok_beto` | el tokenizador de BETO |
| `LARGO_MAX` | 128, el largo máximo de una secuencia, en piezas |
| `IGNORAR` | −100, el valor de etiqueta de las posiciones que la pérdida no cuenta. El Ejercicio 2 explica su uso |

In [ ]:
# ─── Preparación. Las piezas de una oración ya dividida en palabras ─────────
tok_beto = AutoTokenizer.from_pretrained(BETO)
LARGO_MAX, IGNORAR = 128, -100

palabras_ejemplo = val[EJEMPLO][0]
codificada = tok_beto(palabras_ejemplo, is_split_into_words=True)
indices_palabra = codificada.word_ids()
display(pd.DataFrame({
    "pieza": tok_beto.convert_ids_to_tokens(codificada["input_ids"]),
    "word_ids()": pd.Series(indices_palabra, dtype=object),
    "palabra": [palabras_ejemplo[w] if w is not None else "" for w in indices_palabra],
}).T)

largos = [len(x) for x in tok_beto([p for p, _ in ent], is_split_into_words=True)["input_ids"]]
print(f"piezas por oración de entrenamiento, percentil 99: {np.percentile(largos, 99):.0f}")
print(f"oraciones de entrenamiento con más de {LARGO_MAX} piezas: "
      f"{sum(n > LARGO_MAX for n in largos)} de {len(largos)}")

### Ejercicio 2. De palabras a piezas y de piezas a palabras

**Objetivo.** Escribir las dos funciones que trasladan las etiquetas entre las palabras del corpus y las piezas de un tokenizador.

> **Nota. La convención para etiquetar piezas.** El corpus da una sola etiqueta por palabra. El modelo produce una predicción por pieza, así que una palabra partida en tres piezas recibe tres predicciones. La convención habitual pone la etiqueta de la palabra en la posición de su **primera pieza**. Las demás piezas de la palabra y las posiciones de `[CLS]` y `[SEP]` llevan el valor `IGNORAR`, que es −100. La función `F.cross_entropy` tiene un argumento `ignore_index`, cuyo valor por defecto es −100, y las posiciones cuya etiqueta vale `ignore_index` no intervienen en la pérdida ni en su promedio. Con esa convención, la pérdida cuenta exactamente una predicción por palabra. Para volver a las palabras después de predecir, la etiqueta de cada palabra es la que el modelo predice en la posición de su primera pieza.

**Enunciado:**

1. Escribí la función `alinear(oraciones, tok)`. El argumento `oraciones` es una lista de pares `(palabras, etiquetas)`, como `val`. El argumento `tok` es un tokenizador.

   Dentro de la función, tokenizá todas las oraciones en una sola llamada a `tok`, con los argumentos `is_split_into_words=True`, `truncation=True` y `max_length=LARGO_MAX`. Para ubicar la primera pieza de cada palabra, recorré `word_ids(i)` de cada oración `i`. Una posición es la primera pieza de una palabra si su valor es distinto de `None` y distinto del valor de la posición anterior.

   La función tiene que devolver un diccionario con las claves `"ids"`, `"etiquetas"` y `"palabra_de"`. El valor de cada clave es una lista con un elemento por oración, y cada elemento es una lista con un valor por posición de la secuencia de piezas. Completá cada posición así:
   - en `"ids"`, el índice de la pieza, que el tokenizador devuelve en `input_ids`;
   - en `"etiquetas"`, el índice de etiqueta de la palabra si la posición es su primera pieza, e `IGNORAR` en las demás;
   - en `"palabra_de"`, el índice de la palabra si la posición es su primera pieza, y `None` en las demás.

2. Escribí la función `a_palabras(pred, palabra_de, n_palabras)`. Los tres argumentos corresponden a una sola oración:
   - `pred` es la lista de etiquetas predichas, con un índice de etiqueta por posición de la secuencia de piezas;
   - `palabra_de` es la lista que devuelve `alinear` para esa oración, del mismo largo que `pred`;
   - `n_palabras` es la cantidad de palabras de la oración.

   La función tiene que devolver una lista con un índice de etiqueta por palabra. Tomá la etiqueta de cada palabra de `pred`, en la posición de su primera pieza. A las palabras que no tienen ninguna posición, porque la oración se truncó, asignales `0`. La celda de preparación del Ejercicio 3 usa esta función para convertir las predicciones de todos los modelos del laboratorio.

3. Creá un diccionario llamado `datos_beto`, con las claves `"ent"`, `"val"` y `"prueba"`. El valor de cada clave es el resultado de `alinear` sobre la partición correspondiente de `particiones`, con `tok_beto`.

4. Verificá las dos funciones.
   - Mostrá, para la oración `val[EJEMPLO]`, una tabla con una columna por posición y dos filas, la pieza y la etiqueta alineada, como la tabla de la preparación. La etiqueta se muestra con su nombre, o con el texto `IGNORAR`.
   - Aplicá `a_palabras` a las etiquetas alineadas de cada oración de validación, como si fueran una predicción, y contá en cuántas oraciones el resultado coincide con las etiquetas originales. Imprimí ese número junto con la cantidad de oraciones de validación que tienen alguna palabra sin posición en su `palabra_de`. Solo una oración con alguna palabra sin posición puede no coincidir, así que la cantidad de oraciones que no coinciden no puede superar a la segunda cifra.

> **Pista.** El método `convert_ids_to_tokens` del tokenizador convierte una lista de índices en la lista de piezas.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

La etiqueta de «Palomares» se lee en la posición de `Pal`, la primera de sus tres piezas. Explicá por qué el vector de salida de BETO en esa posición tiene en cuenta también `##oma` y `##res`. Usá lo que muestra la sección 2.5 de la Clase 1 sobre la máscara causal.

*(Escribí tu respuesta acá)*

---
### Preparación. El bucle de afinado

La celda que sigue define el bucle de afinado con la receta de la sección 3.1 de la Clase 2, que es la de `comun.entrenar`. El optimizador es AdamW, con un `lr` de `2e-5`. Durante el primer 10 % de los pasos, el `lr` sube desde cero hasta `2e-5`, que es el calentamiento. Después, el `lr` baja linealmente hasta cero. En la GPU, el bucle usa precisión mixta. Además, el bucle recorta la norma del gradiente en 1, como en la Clase 2 de la Unidad 2. Los lotes se construyen con oraciones de largo parecido, como en la Parte 3 de la Clase 2, para reducir el relleno.

El bucle difiere del de la Clase 2 en tres puntos.

- **Cada lote lleva una etiqueta por posición.** `construir_lote` rellena las etiquetas con `IGNORAR`. El índice de relleno de BETO, `tok_beto.pad_token_id`, vale 1, que en `ETIQUETAS` es `B-PER`. Con las etiquetas rellenadas con ese valor, la pérdida le enseñaría al modelo a predecir `B-PER` en las posiciones de relleno.
- **La pérdida es un argumento de `entrenar`.** El laboratorio afina dos clases de modelo. Cada clase calcula la pérdida de una manera distinta. `BertForTokenClassification` calcula la pérdida dentro de su `forward` cuando recibe las etiquetas. La clase que escribís en el Ejercicio 3 devuelve solo los logits. La pérdida de esa clase la calcula la función `perdida_etiquetador`, que también escribís en ese ejercicio. Por eso `entrenar` recibe una función `perdida_de(modelo, lote)`, que devuelve la pérdida del lote. Para el modelo de Hugging Face, esa función es `perdida_hf`, definida en esta celda.
- **Son tres épocas.** La Clase 2 afinó una época sobre 100.000 reseñas en lotes de 32, que son 3.125 pasos. Una época de CoNLL-2002 en lotes de 16 son 521 pasos, y tres épocas son 1.563.

La función que usan los ejercicios es `afinar`. `afinar` fija la semilla, construye el modelo, lo afina con `entrenar` y predice la etiqueta de cada posición de validación y de prueba. Después, `afinar` guarda en `CARPETA` un archivo `.json` con la configuración, la curva de la pérdida, el tiempo de afinado y esas predicciones. Los pesos no se guardan. Si el archivo ya existe y su configuración coincide, `afinar` lo carga y no afina.

La configuración incluye una **huella** del código del modelo y de la pérdida, y otra de los datos alineados. Si corregís tu clase, tu pérdida o `alinear` después de haber afinado un modelo, la huella cambia y `afinar` vuelve a afinarlo, en lugar de cargar un resultado obtenido con el código anterior. Las predicciones se guardan por posición, y `afinar` las convierte a palabras con tu `a_palabras` cada vez que las carga. Una corrección de `a_palabras`, entonces, no requiere volver a afinar.

| Nombre | Qué es |
|---|---|
| `LOTE`, `EPOCAS`, `LR` | 16, 3 y `2e-5` |
| `construir_lote(datos, indices, pad_id)` | la tupla `(ids, mascara, etiquetas)` de las oraciones `indices` de una partición alineada, como `datos_beto["val"]`. Los tres tensores tienen forma `(lote, L)`, donde `L` es el largo de la oración más larga del lote |
| `entrenar(modelo, datos, perdida_de, pad_id, semilla)` | el bucle de afinado sobre una partición alineada, que devuelve la curva de la pérdida, los segundos y la memoria pico |
| `logits_de(modelo, ids, mascara)` | los logits de forma `(lote, L, 9)`, para cualquiera de los dos tipos de modelo |
| `predecir_piezas(modelo, datos, pad_id)` | la etiqueta más probable en cada posición de cada oración de una partición alineada |
| `afinar(nombre, construir, perdida_de, datos, pad_id, semilla)` | la función que afina un modelo o carga su resultado guardado. `construir` es una función sin argumentos que devuelve el modelo sin afinar, y `datos` es un diccionario como `datos_beto`. `afinar` devuelve un diccionario con las claves `"val"` y `"prueba"`, que contienen las etiquetas predichas por palabra con el formato de `pred_O`, y con las claves `"historia"`, `"minutos"` y `"entrenables"` |
| `perdida_hf(modelo, lote)` | la pérdida de `BertForTokenClassification`, que la clase calcula cuando recibe las etiquetas |

In [ ]:
# ─── Preparación. Los lotes, el bucle de afinado y la predicción ────────────
LOTE, EPOCAS, LR = 16, 3, 2e-5


def rellenar(filas, valor):
    """Lista de listas de enteros → tensor (lote, L), rellenado con `valor` hasta la fila más larga."""
    L = max(len(f) for f in filas)
    x = torch.full((len(filas), L), valor, dtype=torch.long)
    for i, f in enumerate(filas):
        x[i, :len(f)] = torch.tensor(f)
    return x.to(DISPOSITIVO)


def construir_lote(datos, indices, pad_id):
    """(ids, máscara, etiquetas) de las oraciones `indices`. Las etiquetas se rellenan con IGNORAR."""
    ids = rellenar([datos["ids"][i] for i in indices], pad_id)
    etiquetas = rellenar([datos["etiquetas"][i] for i in indices], IGNORAR)
    return ids, (ids != pad_id).long(), etiquetas


def lotes_por_largo(datos, lote, semilla):
    """Índices de oraciones agrupados en lotes de largo parecido, con los lotes en orden al azar."""
    gen = np.random.default_rng(semilla)
    largos = np.array([len(x) for x in datos["ids"]]) + gen.random(len(datos["ids"]))
    orden = np.argsort(largos)                       # el sumando al azar desempata los largos iguales
    lotes = [orden[i:i + lote] for i in range(0, len(orden), lote)]
    return [lotes[k] for k in gen.permutation(len(lotes))]


def entrenar(modelo, datos, perdida_de, pad_id, semilla):
    """Afina los parámetros con requires_grad=True, con la receta de la Clase 2 (sección 3.1).

    datos: una partición alineada. perdida_de(modelo, lote) devuelve la pérdida del lote.
    Devuelve la historia de la pérdida, los segundos y la memoria pico de la GPU.
    """
    parametros = [p for p in modelo.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(parametros, lr=LR, weight_decay=0.01)
    pasos = EPOCAS * math.ceil(len(datos["ids"]) / LOTE)
    calentamiento = max(1, pasos // 10)
    programa = torch.optim.lr_scheduler.LambdaLR(   # multiplicador del lr: de 0 a 1, y de 1 a 0
        opt, lambda p: min((p + 1) / calentamiento, max(0.0, (pasos - p) / (pasos - calentamiento))))
    escalador = torch.amp.GradScaler(enabled=EN_GPU)
    if EN_GPU:
        torch.cuda.reset_peak_memory_stats()
    historia, paso, acumulada, t0 = [], 0, 0.0, time.time()
    modelo.train()
    for epoca in range(EPOCAS):
        for indices in lotes_por_largo(datos, LOTE, semilla + epoca):
            lote = construir_lote(datos, indices, pad_id)
            with torch.autocast("cuda" if EN_GPU else "cpu", dtype=torch.float16, enabled=EN_GPU):
                perdida = perdida_de(modelo, lote)
            opt.zero_grad(set_to_none=True)
            escalador.scale(perdida).backward()
            escalador.unscale_(opt)
            nn.utils.clip_grad_norm_(parametros, 1.0)
            escalador.step(opt)
            escalador.update()
            programa.step()
            paso += 1
            acumulada += perdida.item()
            if paso % 250 == 0 or paso == pasos:
                n = paso - (historia[-1]["paso"] if historia else 0)
                historia.append({"paso": paso, "pérdida": round(acumulada / n, 4)})
                print(f"   paso {paso:5d}/{pasos}  pérdida {acumulada / n:.3f}", flush=True)
                acumulada = 0.0
    return {"historia": historia, "segundos": round(time.time() - t0, 1),
            "memoria_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2) if EN_GPU else None}


def logits_de(modelo, ids, mascara):
    """Los logits (lote, L, etiquetas). Las clases de Hugging Face los devuelven en `.logits`."""
    salida = modelo(ids, mascara)
    return getattr(salida, "logits", salida)


def perdida_hf(modelo, lote):
    """La pérdida de BertForTokenClassification, que la clase calcula al recibir `labels`."""
    ids, mascara, etiquetas = lote
    return modelo(input_ids=ids, attention_mask=mascara, labels=etiquetas).loss


@torch.no_grad()
def predecir_piezas(modelo, datos, pad_id, lote=64):
    """La etiqueta más probable en cada posición de cada oración de una partición alineada."""
    modelo.eval()
    salida = [None] * len(datos["ids"])
    orden = np.argsort([len(x) for x in datos["ids"]])
    for i0 in range(0, len(orden), lote):
        indices = orden[i0:i0 + lote]
        ids = rellenar([datos["ids"][i] for i in indices], pad_id)
        with torch.autocast("cuda" if EN_GPU else "cpu", dtype=torch.float16, enabled=EN_GPU):
            pred = logits_de(modelo, ids, (ids != pad_id).long()).argmax(-1).cpu()
        for fila, i in enumerate(indices):
            salida[i] = pred[fila, :len(datos["ids"][i])].tolist()
    return salida


# ─── La huella. Cambia si cambia el código o los datos, no los comentarios ──
def _bytecode(codigo):
    partes = [codigo.co_code, " ".join(codigo.co_names).encode()]
    for k in codigo.co_consts:
        partes.append(_bytecode(k) if hasattr(k, "co_code") else repr(k).encode())
    return b"".join(partes)


def huella(*funciones):
    """Resumen del bytecode de las funciones, de los métodos de las clases del notebook
    que nombran y de los textos globales que usan, como el nombre de un modelo."""
    vistas, partes, pendientes = set(), [], list(funciones)
    while pendientes:
        f = pendientes.pop(0)
        if id(f) in vistas:
            continue
        vistas.add(id(f))
        if isinstance(f, type):
            pendientes += [v for v in vars(f).values() if callable(v)]
            continue
        codigo = getattr(f, "__code__", None)
        if codigo is None:
            continue
        partes += [_bytecode(codigo), repr(getattr(f, "__defaults__", None)).encode()]
        for nombre in codigo.co_names:
            valor = f.__globals__.get(nombre)
            if isinstance(valor, str):
                partes.append(valor.encode())
            elif isinstance(valor, type) and valor.__module__ == "__main__":
                pendientes.append(valor)
    return hashlib.sha256(b"".join(partes)).hexdigest()[:16]


def huella_datos(datos):
    texto = json.dumps([datos["ent"]["ids"], datos["ent"]["etiquetas"], datos["val"]["ids"]])
    return hashlib.sha256(texto.encode()).hexdigest()[:16]


# ─── Afinar, o cargar el resultado guardado ────────────────────────────────
def afinar(nombre, construir, perdida_de, datos, pad_id, semilla):
    """Afina el modelo que devuelve construir() y predice validación y prueba.

    Guarda en CARPETA un .json con la configuración, la historia, el tiempo y las
    predicciones por posición. No guarda los pesos. Si el .json existe y la
    configuración coincide, lo carga y no afina.
    Devuelve un diccionario con "val" y "prueba" (etiquetas predichas por palabra, una
    lista por oración), "historia", "minutos" y "entrenables".
    """
    config = {"semilla": semilla, "lote": LOTE, "epocas": EPOCAS, "lr": LR,
              "largo_max": LARGO_MAX, "codigo": huella(construir, perdida_de),
              "datos": huella_datos(datos)}
    ruta = os.path.join(CARPETA, f"{nombre}.json")
    r = None
    if os.path.exists(ruta):
        with open(ruta, encoding="utf-8") as archivo:
            guardado = json.load(archivo)
        if guardado["config"] == config:
            r = guardado
            print(f"[{nombre}] cargado de {ruta} (afinado el {r['fecha']})")
        else:
            print(f"[{nombre}] cambió la configuración, el código o los datos: se vuelve a afinar")
    if r is None:
        torch.manual_seed(semilla)
        modelo = construir().to(DISPOSITIVO)
        print(f"[{nombre}] afinando {EPOCAS} épocas en {DISPOSITIVO}", flush=True)
        r = entrenar(modelo, datos["ent"], perdida_de, pad_id, semilla)
        r.update(config=config, fecha=time.strftime("%Y-%m-%d %H:%M"),
                 entrenables=sum(p.numel() for p in modelo.parameters() if p.requires_grad),
                 piezas_val=predecir_piezas(modelo, datos["val"], pad_id),
                 piezas_prueba=predecir_piezas(modelo, datos["prueba"], pad_id))
        del modelo
        if EN_GPU:
            torch.cuda.empty_cache()
        with open(ruta, "w", encoding="utf-8") as archivo:
            json.dump(r, archivo)
        print(f"[{nombre}] {r['segundos'] / 60:.1f} minutos, guardado en {ruta}")
    for clave in ("val", "prueba"):
        r[clave] = [a_palabras(pred, palabra_de, len(palabras))
                    for pred, palabra_de, (palabras, _) in
                    zip(r[f"piezas_{clave}"], datos[clave]["palabra_de"], particiones[clave])]
    r["minutos"] = round(r["segundos"] / 60, 1)
    return r

### Ejercicio 3. El etiquetador sobre BETO

**Objetivo.** Escribir el etiquetador sobre el encoder de BETO y su pérdida, y compararlo con la clase equivalente de Hugging Face.

`ClasificadorBERT`, en la sección 3.1 de la Clase 2, aplica su capa lineal al vector de la posición `[CLS]`, que es `h[:, 0]`. El etiquetador aplica la misma capa al vector de cada posición. Como `nn.Linear` opera sobre la última dimensión de su entrada, la capa transforma la salida del encoder, de forma `(lote, L, 768)`, en logits de forma `(lote, L, 9)` sin recorrer las posiciones.

**Enunciado:**

1. Escribí la clase `EtiquetadorBERT`, que herede de `nn.Module`.
   - Su método `__init__(self, nombre=BETO, n_etiquetas=len(ETIQUETAS))` tiene que crear tres atributos, con los mismos nombres que en `ClasificadorBERT`.
     - `encoder` es el `BertModel` preentrenado que indica `nombre`. Va sin la capa *pooler*, porque esa capa transforma el vector de `[CLS]` y el etiquetador no lo usa.
     - `dropout` es un `nn.Dropout` con probabilidad 0,1.
     - `cabeza` es una `nn.Linear` que va del ancho del encoder a `n_etiquetas`. El ancho está en `encoder.config.hidden_size`.

     El argumento `nombre` permite construir en el Ejercicio 4 el mismo etiquetador sobre el BERT multilingüe.
   - Su método `forward(self, ids, mascara)` tiene que devolver los logits de todas las posiciones, de forma `(lote, L, n_etiquetas)`.
2. Escribí la función `perdida_etiquetador(modelo, lote)`, que devuelve la pérdida del etiquetador sobre un lote.
   - `modelo` es un `EtiquetadorBERT`.
   - `lote` es la tupla `(ids, mascara, etiquetas)` que devuelve `construir_lote`.

   La función tiene que calcular los logits de `modelo` y devolver la entropía cruzada con `ignore_index=IGNORAR`. `F.cross_entropy` espera los logits con forma `(N, C)` y las etiquetas con forma `(N,)`, donde N es la cantidad de predicciones y C la de clases. Los logits del etiquetador tienen forma `(lote, L, 9)`, así que la función tiene que unir las dimensiones del lote y de la posición en una sola, con N = lote × L, como la pérdida del modelo de lenguaje de la Clase 2 de la Unidad 2.
3. Verificá la clase y la pérdida.
   - Fijá la semilla de PyTorch en 0, creá un `EtiquetadorBERT` en una variable llamada `demo` y llevalo a `DISPOSITIVO`.
   - Construí con `construir_lote` un lote con las oraciones 0, 1, 2 y 3 de `datos_beto["val"]` y el índice de relleno `tok_beto.pad_token_id`.
   - Imprimí la forma de los logits, la cantidad de posiciones del lote, la cantidad de posiciones cuya etiqueta es distinta de `IGNORAR` y el valor de la pérdida. Con la cabeza sin afinar, la pérdida tiene que estar entre 2 y 3. El valor de referencia es $\ln 9 \approx 2{,}2$, la entropía cruzada de una distribución uniforme sobre las nueve etiquetas, y la pérdida no coincide con él porque la cabeza sin afinar no produce nueve logits iguales.
   - Al terminar, borrá el modelo con `del demo`, para liberar la memoria de la GPU antes de los afinados.

In [ ]:
# Tu código aquí

### Preparación. Los cuatro afinados sobre BETO

La celda que sigue afina cuatro modelos sobre BETO con `afinar`. Los cuatro resultan de combinar dos implementaciones con dos semillas. Las implementaciones son `BertForTokenClassification`, la clase de Hugging Face con la misma arquitectura que tu etiquetador, y tu `EtiquetadorBERT`. Las semillas son 0 y 1. La semilla fija la inicialización de la cabeza, las máscaras del *dropout* y el orden de los lotes. Las dos semillas permiten medir cuánto varía el F1 cuando solo cambia la semilla. Esa variación es la escala con la que se juzgan las diferencias de los dos ejercicios siguientes. El Ejercicio 4 la usa para la diferencia entre BETO y el multilingüe, y el Ejercicio 5 para la diferencia entre el F1 de todas las oraciones de prueba y el de las que no aparecen en entrenamiento.

`BertForTokenClassification` agrega al encoder un *dropout* y una capa lineal aplicada a todas las posiciones, igual que tu clase. Las dos difieren en la inicialización de la capa lineal. `transformers` inicializa los pesos con una distribución normal de desvío 0,02 y los sesgos en cero, y `nn.Linear` usa por defecto una distribución uniforme.

Al terminar, la celda construye la tabla `comparacion`, con una fila por modelo y cuatro columnas. Las columnas son el F1 de validación, el F1 de prueba, los minutos de afinado y los parámetros entrenables en millones. El Ejercicio 4 le agrega una fila.

El F1 de validación de un etiquetador con las etiquetas bien alineadas supera 80. Una alineación equivocada no produce ningún error de ejecución, porque el modelo se afina igual sobre etiquetas ubicadas en posiciones que no les corresponden. Si `BertForTokenClassification` y tu clase quedan por debajo de 80, el error está en `alinear` o en `a_palabras`, que los dos modelos comparten. Si solo tu clase queda por debajo de 80, el error está en `EtiquetadorBERT` o en `perdida_etiquetador`.

| Nombre | Qué es |
|---|---|
| `resultados` | un diccionario con lo que devuelve `afinar` para cada modelo, con las claves `"hf_s0"`, `"mano_s0"`, `"hf_s1"` y `"mano_s1"` |
| `resumen(r)` | la fila de `comparacion` que corresponde a un resultado `r` de `afinar` |
| `comparacion` | la tabla de los cuatro modelos |

In [ ]:
# ─── Preparación. Cuatro afinados: dos implementaciones, dos semillas ───────
def construir_hf():
    return BertForTokenClassification.from_pretrained(BETO, num_labels=len(ETIQUETAS))


def construir_mano():
    return EtiquetadorBERT()


PLAN = [("hf_s0", construir_hf, perdida_hf, 0), ("mano_s0", construir_mano, perdida_etiquetador, 0),
        ("hf_s1", construir_hf, perdida_hf, 1), ("mano_s1", construir_mano, perdida_etiquetador, 1)]
resultados = {nombre: afinar(nombre, construir, perdida, datos_beto, tok_beto.pad_token_id, semilla)
              for nombre, construir, perdida, semilla in PLAN}


def resumen(r):
    """La fila de la tabla comparativa para un resultado de afinar."""
    return pd.Series({"F1 val.": medir(val, r["val"])["F1"],
                      "F1 prueba": medir(prueba, r["prueba"])["F1"],
                      "minutos": r["minutos"],
                      "parámetros entrenables (M)": round(r["entrenables"] / 1e6, 1)})


NOMBRES = {"hf_s0": "BertForTokenClassification, semilla 0", "mano_s0": "a mano, semilla 0",
           "hf_s1": "BertForTokenClassification, semilla 1", "mano_s1": "a mano, semilla 1"}
comparacion = pd.DataFrame({NOMBRES[k]: resumen(r) for k, r in resultados.items()}).T
comparacion

### Preparación. Las apariciones de «Madrid» en validación

La celda que sigue busca la palabra «Madrid» en las oraciones de validación y agrupa sus apariciones según la etiqueta de referencia. Para cada etiqueta, la celda cuenta en cuántas apariciones acierta la línea de base de memorización del Ejercicio 1 y en cuántas acierta tu `EtiquetadorBERT` afinado con la semilla 0. La pregunta de análisis del Ejercicio 3 usa esa tabla.

| Nombre | Qué es |
|---|---|
| `madrid` | la tabla, con una fila por etiqueta de referencia y tres columnas, que son la cantidad de apariciones, los aciertos de la memorización y los aciertos del etiquetador |

In [ ]:
# ─── Preparación. «Madrid» en validación, memorización contra etiquetador ───
por_etiqueta = {}
for s, (palabras, etq) in enumerate(val):
    for i, palabra in enumerate(palabras):
        if palabra != "Madrid":
            continue
        fila = por_etiqueta.setdefault(ETIQUETAS[etq[i]], Counter())
        fila["apariciones"] += 1
        fila["aciertos de la memorización"] += pred_memoria[s][i] == etq[i]
        fila["aciertos del etiquetador"] += resultados["mano_s0"]["val"][s][i] == etq[i]
madrid = pd.DataFrame(por_etiqueta).T.fillna(0).astype(int).sort_values("apariciones", ascending=False)
madrid.index.name = "etiqueta de referencia"
madrid

**Pregunta de análisis:**

En entrenamiento, «Madrid» aparece con cinco etiquetas distintas, que imprimió el Ejercicio 1. Las dos más frecuentes corresponden a dos entidades distintas. Sola, «Madrid» es una entidad `LOC`, la ciudad, con la etiqueta `B-LOC`. En «Real Madrid», «Madrid» es la segunda palabra de una entidad `ORG`, el club, y lleva la etiqueta `I-ORG`. Con lo que muestra la sección 2.3 de la Clase 2 sobre las representaciones contextuales, explicá qué información usa el etiquetador para distinguir esos dos casos y por qué la memorización no puede usarla. Apoyá la explicación en la tabla `madrid`.

*(Escribí tu respuesta acá)*

### Ejercicio 4. BETO y el BERT multilingüe

**Objetivo.** Afinar el mismo etiquetador sobre el BERT multilingüe, y comparar su diferencia con BETO con la que midió la Clase 2 sin afinar.

La sección 2.1 de la Clase 2 midió los dos modelos en la tarea de palabras tapadas, sin afinar, sobre las mismas 3.424 palabras de reseñas de prueba. BETO acertó la palabra tapada en el 51,0 % de los casos, y el BERT multilingüe, que reparte su vocabulario entre 104 idiomas, en el 34,1 %. Este ejercicio repite el afinado de `EtiquetadorBERT` con la semilla 0 y cambia un solo **factor** del montaje, el modelo preentrenado. El tokenizador cambia con el modelo. Los índices de las piezas son de otro vocabulario, y una misma palabra se divide en piezas distintas, así que la primera pieza de cada palabra queda en otra posición. Por eso las oraciones se vuelven a alinear con el tokenizador del multilingüe.

**Enunciado:**

1. Creá el tokenizador del multilingüe con `AutoTokenizer.from_pretrained` y el nombre `MULTI`, en una variable llamada `tok_multi`. Creá además un diccionario llamado `datos_multi`, con las mismas claves que `datos_beto`. El valor de cada clave es el resultado de `alinear` sobre la partición correspondiente de `particiones`, con `tok_multi`.
2. Afiná el etiquetador sobre el multilingüe con `afinar`, con el nombre `"multi_s0"`, la pérdida `perdida_etiquetador`, los datos `datos_multi`, el índice de relleno de `tok_multi` y la semilla 0. La función que construye el modelo no recibe argumentos y devuelve `EtiquetadorBERT(MULTI)`. Guardá el resultado en una variable llamada `multi_s0`, y agregalo a `comparacion` como una fila nueva con el índice `"multilingüe, a mano, semilla 0"`, construida con `resumen`. Mostrá `comparacion`.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

Decí si para esta tarea conviene afinar BETO o el multilingüe. Justificalo con la diferencia de F1 de validación entre los dos modelos, con la variación entre semillas del Ejercicio 3 como escala, y con las demás columnas de `comparacion`. Compará esa diferencia con la que midió la sección 2.1 de la Clase 2 sin afinar, en la tarea de palabras tapadas.

*(Escribí tu respuesta acá)*

### Ejercicio 5. Las particiones del corpus

**Objetivo.** Verificar si las oraciones de validación y de prueba aparecen también en la partición de entrenamiento, y medir el etiquetador por separado sobre las que aparecen y sobre las que no.

Una partición de evaluación mide la capacidad de generalizar de un modelo si sus ejemplos no están en la partición de entrenamiento. Las particiones de CoNLL-2002 vienen fijadas por el corpus. Este ejercicio verifica esa condición comparando oraciones completas. Una oración de evaluación es **repetida** si `ent` contiene una oración con exactamente las mismas palabras, y es **nueva** en caso contrario. El ejercicio no afina ningún modelo, porque usa las predicciones que `afinar` ya guardó.

**Enunciado:**

1. Creá un conjunto llamado `textos_ent` con el texto de cada oración de `ent`, formado por sus palabras unidas por un espacio. Con ese conjunto se decide si una oración de evaluación es repetida.

   El método `join` de un texto une una lista de textos y pone ese texto entre cada par, así que `" ".join(["Juan", "vive"])` da `"Juan vive"`.

   Un conjunto, de tipo `set` en Python, guarda elementos sin repetir y sin orden. Se construye con `set(...)` sobre cualquier secuencia, o con llaves, como `{x for x in secuencia}`. El ejercicio usa el operador `in`. La expresión `texto in textos_ent` vale `True` si `texto` está en el conjunto, y en un conjunto tarda lo mismo cualquiera sea su tamaño. En una lista, `in` compara contra los elementos uno por uno, y `ent` tiene 8.324 oraciones.
2. Construí un `DataFrame` llamado `solapamiento`, con una columna para `val` y otra para `prueba`, y estas cinco filas:
   - el porcentaje de oraciones repetidas;
   - el porcentaje de las entidades de referencia de la partición que están en oraciones repetidas. La cantidad de entidades de una oración es el largo de la lista que devuelve `entidades` con sus etiquetas de referencia;
   - el F1 de `EtiquetadorBERT` con la semilla 0 sobre todas las oraciones de la partición;
   - el mismo F1 sobre las oraciones repetidas;
   - el mismo F1 sobre las oraciones nuevas.

   Las predicciones están en `resultados["mano_s0"]`, en las claves `"val"` y `"prueba"`, en el mismo orden que las oraciones. Calculá cada F1 con `medir`, pasándole las oraciones de ese grupo y sus predicciones, tomadas en las mismas posiciones para que sigan emparejadas. Mostrá la tabla con un decimal.
3. Imprimí las cinco oraciones repetidas más frecuentes de `val`, cada una con la cantidad de veces que aparece en `val`.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

Compará el F1 de prueba sobre todas las oraciones con el F1 sobre las oraciones nuevas, con la variación entre semillas del Ejercicio 3 como escala. Explicá el tamaño de esa diferencia con las dos primeras filas de `solapamiento`, y decí qué F1 de prueba reportarías.

*(Escribí tu respuesta acá)*

---
## Antes de entregar

Revisá esta lista:

- [ ] Reinicié el entorno y ejecuté **todas** las celdas de arriba a abajo sin errores (**Entorno de ejecución > Reiniciar y ejecutar todo**).
- [ ] En la verificación del Ejercicio 2, la cantidad de oraciones de validación que no coinciden no supera la de oraciones con alguna palabra sin posición.
- [ ] La pérdida de `demo` sin afinar, en el Ejercicio 3, está entre 2 y 3.
- [ ] `comparacion` tiene cinco filas, y la del multilingüe se afinó con `datos_multi` y el índice de relleno de `tok_multi`.
- [ ] En `comparacion`, el F1 de validación de los cinco modelos es mayor que 80.
- [ ] Respondí las preguntas de análisis de los cinco ejercicios.
- [ ] No modifiqué ninguna celda fuera de las de actividad.

---
## Cierre

Este laboratorio construyó la cabeza de etiquetado que la sección 1.3 de la Clase 2 mostraba en la figura del artículo de BERT. El modelo ocupa diez líneas. El resto del trabajo estuvo antes y después del modelo, en el traslado de las etiquetas entre palabras y piezas y en la elección de una métrica que cuenta entidades.

Tres prácticas del laboratorio se aplican a cualquier experimento con modelos preentrenados:

- medir una línea de base antes del modelo, con la misma métrica;
- juzgar la diferencia entre dos variantes con la variación entre semillas como escala;
- verificar, antes de reportar una cifra, si las particiones de evaluación comparten ejemplos con la de entrenamiento.

La Parte B usa el mismo encoder para responder preguntas sobre un texto. La entrada lleva la pregunta y el párrafo separados por `[SEP]`, y la cabeza produce dos puntajes por posición, uno para el comienzo de la respuesta y otro para el final.